Створення вибірки
---

Дані для вибірки взято з Winsconsin Fruit[[1](https://fruit.wisc.edu/2021/09/02/ranges-of-fruit-quality-parameters-for-cold-climate-grape-cultivars-at-harvest/#:~:text=consideration%20these%20factors.-,Table%201,-.%20Fruit%20quality%20parameters)]

|Змінна|Назва|Оптимальне значення|
|-|-|-|
|kind|Сорт винограду|Білий (0) або червоний (1)|
|tss|Кількість розчинних твердих речовин|14-15% = гарно, 17% і вище = добре|
|ta|Змінна кислотність|6-9 г/л|
|ph|Справжня кислотність|2.8-4.0|


In [ ]:
from collections import namedtuple
import random
import pandas as pd

# Визначити кількість екземплярів
rows = 12

# Перекладач сортів винограду
# 0 = білий, 1 = червоний
grape_kinds = {0: "white", 1: "red"}

# Визначити шаблон екземпляру
Grape = namedtuple("Grape", "kind,tss,ta,ph")

# Створити вибірку
grapes = [
    Grape(
        # Сорт або білий або червоний
        kind=random.choice([0, 1]),
        # Кількість розчинних твердих від 17 до 27
        tss=random.randint(17, 27),
        # Змінна кислотність від 6 до 17
        ta=random.randint(6, 17),
        # Кислотність від 2.8 до 4.0 з одним знаком після коми
        ph=round(random.uniform(2.8, 4.0), 1),
    )
    # Стільки разів, скільки написано в змінній rows
    for row in range(rows)
]

# Перетворити вибірку на таблицю даних
df = pd.DataFrame(grapes, columns=Grape._fields)

# Вивести таблицю
df

,kind,tss,ta,ph
0,1,24,14,3.4
1,1,22,8,3.1
2,1,27,6,3.2
3,0,19,6,3.8
4,1,25,17,3.9
5,1,22,12,2.9
6,1,19,12,3.5
7,1,19,11,3.0
8,0,17,10,3.8
9,0,27,9,2.8


Нормалізація вибірки
---

Обрано метод z-нормалізації[[2](https://www.statology.org/z-score-normalization/)]:

$$
new=\frac{value-\mu}{\sigma}
$$

- $new$: нове значення 
- $value$: поточне значення 
- $\mu$: середнє значення ознаки
- $\sigma$: стандартне відхилення ознаки

Щоб отримати стандартне відхилення[[3](https://www.mathsisfun.com/data/standard-deviation.html#:~:text=only%20a%20sample.-,Formulas,-Here%20are%20the)]:

$$
\sigma=\sqrt{\frac{1}{N}\sum_{n=1}^{N}(feature_n-\mu)^2}
$$

- $N$: кількість екземплярів
- $feature$: поточна ознака

Формула середнього значення $\mu$:

$$
\mu=\frac{1}{N}\sum_{n=1}^{N}feature_n
$$

In [290]:
import math


def get_mean(data: list[int]) -> float:
    # Порахувати суму всіх значень
    sum_of_data = sum(data)

    # Повернути суму поділену на кількість значень
    return sum_of_data / len(data)


def get_standard_deviation(data: list) -> float:
    # Отримати середнє значення ознаки
    mean = get_mean(data=data)

    # Порахувати різниці між кожним елементом і середнім значенням
    # Піднести до квадрату
    squares = [(element - mean) ** 2 for element in data]

    # Отримати середнє значення з порахованих різниць
    average = get_mean(squares)

    # Повернути квадратний корінь середнього значення
    return math.sqrt(average)


for column_name, column_data in df.items():
    # Пропустити сорт винограду
    if column_name == "kind":
        continue

    # Перетворити цілочисельні на плаваючі коми
    df[column_name] = df[column_name].astype(float)

    # Отримати середнє значення ознаки та стандартне відхилення
    column_mean = get_mean(column_data.to_list())
    column_std = get_standard_deviation(column_data.to_list())

    # Отримати номер екземпляру (index) та його значення (value)
    for row_index, row_value in enumerate(column_data):
        # Порахувати різницю між цим значенням і середнім по всій ознаці
        numerator = row_value - column_mean

        # Поділити різницю та стандартне відхилення ознаки
        result = numerator / column_std

        # Змінити це значення в таблиці на пораховане
        df.at[row_index, column_name] = result

# Вивести таблицю
df

,kind,tss,ta,ph
0,1,0.521286,0.970855,0.124035
1,1,-0.104257,-0.868659,-0.620174
2,1,1.459601,-1.481831,-0.372104
3,0,-1.042572,-1.481831,1.116313
4,1,0.834058,1.890612,1.364382
5,1,-0.104257,0.357683,-1.116313
6,1,-1.042572,0.357683,0.372104
7,1,-1.042572,0.051098,-0.868243
8,0,-1.668115,-0.255488,1.116313
9,0,1.459601,-0.562074,-1.364382
